# <font color="#4a86e8"> **Piper Training: Python 3.10 Edition (Conda)**

**Why this works:**
You asked for **Python 3.10**. This notebook installs **Miniconda** directly inside Colab and creates a clean Python 3.10 environment.

**Benefits:**
1.  **NO Compilation Errors:** Piper provides pre-built wheels for Python 3.10. It installs instantly.
2.  **Stable:** We use the exact Python version the developers use.

**How it works:**
Instead of running python code directly in the cells, we send commands to our special `piper_env`. 

---

### **Instructions:**
1.  Upload `TextyMcSpeechy.zip` to Drive.
2.  Run all cells.

In [ ]:
#@markdown # <font color="#4a86e8"> **1. Install Python 3.10 (Miniconda)** 🐍
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# 1. Install Miniconda (if not already present)
if not os.path.exists("/usr/local/bin/conda"):
    print("Installing Miniconda...")
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
    !bash Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local > /dev/null
    print("Miniconda installed.")
else:
    print("Miniconda already installed.")

# 2. Create Python 3.10 Environment
# We strictly use 3.10 to match Piper's wheels
print("Creating 'piper_env' with Python 3.10...")
!conda create -y -n piper_env python=3.10

# 3. Install System Deps
!sudo apt-get update -y > /dev/null
!sudo apt-get install -y espeak-ng portaudio19-dev libespeak-ng-dev > /dev/null

print("✅ Python 3.10 Environment Ready!")

In [ ]:
#@markdown # <font color="#4a86e8"> **2. Install Piper (Fast & Easy)** ⚡
# Note: We use 'conda run -n piper_env' to execute commands inside our custom 3.10 environment

# 1. Install Piper directly (No build needed for 3.10!)
print("Installing dependencies in Python 3.10 environment...")

# Install PyTorch Lightning 1.9.x (Compatible)
!conda run -n piper_env pip install "pytorch-lightning==1.9.5" "torchmetrics>=0.7.0"

# Install Piper Phonemize (Pre-built wheel exists for 3.10!)
!conda run -n piper_env pip install piper-phonemize

# Install Piper Train from source
%cd /content
if not os.path.exists("piper"):
    !git clone https://github.com/rhasspy/piper.git

# Install requirements inside the env
%cd /content/piper/src/python
!conda run -n piper_env pip install -r requirements.txt
!conda run -n piper_env pip install -e .

# Build Monotonic Align (Still needed, but easy on 3.10)
!conda run -n piper_env ./build_monotonic_align.sh

print("✅ Piper Installed successfully on Python 3.10!")

In [ ]:
#@markdown # <font color="#4a86e8"> **3. Prepare Dataset (Local)** 📂
import os
import shutil
import zipfile

DRIVE_ROOT = "/content/drive/MyDrive"
TEXTY_DIR = os.path.join(DRIVE_ROOT, "TextyMcSpeechy")
TEXTY_ZIP = os.path.join(DRIVE_ROOT, "TextyMcSpeechy.zip")
SOURCE_DATASET = os.path.join(TEXTY_DIR, "tts_dojo/DATASETS/tamil_dataset")
WORKSPACE_DIR = "/content/workspace"
DATASET_DIR = os.path.join(WORKSPACE_DIR, "dataset")

# Cleanup
if os.path.exists(WORKSPACE_DIR):
    shutil.rmtree(WORKSPACE_DIR)
os.makedirs(DATASET_DIR)
os.makedirs(os.path.join(DATASET_DIR, "wavs"))

# Unzip
if not os.path.exists(TEXTY_DIR):
    if os.path.exists(TEXTY_ZIP):
        with zipfile.ZipFile(TEXTY_ZIP, 'r') as zip_ref:
            zip_ref.extractall(DRIVE_ROOT)

# Copy Data
print("Copying dataset files locally...")
shutil.copy(os.path.join(SOURCE_DATASET, "metadata.csv"), os.path.join(DATASET_DIR, "metadata.csv"))

wav_src = os.path.join(SOURCE_DATASET, "wav_22050")
if not os.path.exists(wav_src):
    wav_src = os.path.join(SOURCE_DATASET, "wavs")
    
!cp -r "{wav_src}"/* "{DATASET_DIR}/wavs/"

print("✅ Data Ready.")

In [ ]:
#@markdown # <font color="#4a86e8"> **4. Preprocess & Train (Run in Py3.10)** 🚀
import os

WORKSPACE_DIR = "/content/workspace"
OUTPUT_DIR = os.path.join(WORKSPACE_DIR, "piper_train_tamil")
DRIVE_BACKUP = "/content/drive/MyDrive/piper_train_tamil/checkpoints"

if not os.path.exists(DRIVE_BACKUP):
    os.makedirs(DRIVE_BACKUP)

# 1. Preprocess
print("Running Preprocess...")
!conda run -n piper_env python -m piper_train.preprocess \
  --language ta \
  --input-dir "{WORKSPACE_DIR}/dataset" \
  --output-dir "{OUTPUT_DIR}" \
  --dataset-name "tamil_piper" \
  --dataset-format ljspeech \
  --sample-rate 22050

# 2. Train
print("Starting Training...")
!conda run -n piper_env python -m piper_train \
    --dataset-dir "{OUTPUT_DIR}" \
    --accelerator gpu \
    --devices 1 \
    --batch-size 8 \
    --validation-split 0.0 \
    --num-test-examples 0 \
    --max_epochs 100 \
    --checkpoint-epochs 5 \
    --precision 32 \
    --quality medium \
    --callbacks.default_checkpoint_monitor.dirpath "{DRIVE_BACKUP}"

!cp -r "{OUTPUT_DIR}/lightning_logs" "/content/drive/MyDrive/piper_train_tamil/"